# Извлечение эмбеддингов

In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor

In [2]:
DATA_DIR = Path("../data")
SRC_DIR = DATA_DIR / "filtered_by_images"
IMAGES_DIR = DATA_DIR / "images"
OUT_DIR = DATA_DIR / "embeddings_siglip2"
SHARD_DIR = OUT_DIR / "shards"
FINAL_PATH = OUT_DIR / "item_embeddings.parquet"

SHARD_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
MODEL_NAME = "google/siglip2-base-patch16-256"
SPLITS = ["train", "val", "test"]

MAX_ITEMS = None       # None = все; 1000 для теста
CHUNK_SIZE = 20_000    # размер шарда-чекпойнта
TEXT_BATCH = 128
IMAGE_BATCH = 64

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32

## Модель и энкодеры

In [4]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
TEXT_MAX_LEN = model.config.text_config.max_position_embeddings

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

In [5]:
def _features(out: torch.Tensor | object) -> torch.Tensor:
    if isinstance(out, torch.Tensor):
        return out
    if getattr(out, "pooler_output", None) is not None:
        return out.pooler_output
    return out.last_hidden_state[:, 0]


@torch.inference_mode()
def encode_texts(texts: list[str]) -> np.ndarray:
    inputs = processor(
        text=texts,
        return_tensors="pt",
        padding=True,
        padding_side="right",
        truncation=True,
        max_length=TEXT_MAX_LEN,
    ).to(DEVICE)
    embs = _features(model.get_text_features(**inputs))
    return F.normalize(embs.float(), dim=-1).cpu().numpy()


@torch.inference_mode()
def encode_images(images: list[Image.Image]) -> np.ndarray:
    inputs = processor(images=images, return_tensors="pt", padding=True).to(DEVICE)
    embs = _features(model.get_image_features(**inputs))
    return F.normalize(embs.float(), dim=-1).cpu().numpy()


# probe dim
_probe = encode_texts(["test probe"])
EMB_DIM = _probe.shape[1]
print(f"model={MODEL_NAME}, device={DEVICE}, dim={EMB_DIM}")

model=google/siglip2-base-patch16-256, device=cuda, dim=768


## Каталог айтемов с заголовками

In [ ]:
items = (
    pl.concat(
        [pl.scan_parquet(SRC_DIR / f"{s}.parquet").select("item_id", "title") for s in SPLITS]
    )
    .unique(subset="item_id", keep="first")
    .sort("item_id")
    .collect()
)

# проверка работоспособности
if MAX_ITEMS is not None:
    items = items.head(MAX_ITEMS)

print(f"Айтемов к обработке: {items.height:,}")
items.head()

Айтемов к обработке: 2,903,192


item_id,title
i64,str
172307,"""Колодец Септик Выгребная яма С…"
200074,"""Водитель категории Е по городу"""
214791,"""Свободного назначения, 69 м²"""
224370,"""Аллиумы"""
234190,"""Магнитола Навигация Ситроен Пе…"


## Методы для извлечения эмбеддингов по батчам

In [11]:
# Метод получения пути к изображению
def image_path(item_id: int) -> Path:
    return IMAGES_DIR / f"{item_id % 1000:03d}" / f"{item_id}.jpg"


def encode_texts_batched(texts: list[str]) -> np.ndarray:
    parts = [encode_texts(texts[i : i + TEXT_BATCH]) for i in range(0, len(texts), TEXT_BATCH)]
    return np.vstack(parts)


def encode_images_batched(paths: list[Path]) -> tuple[np.ndarray, np.ndarray]:
    n = len(paths)
    out = np.zeros((n, EMB_DIM), dtype=np.float32)
    ok = np.zeros(n, dtype=bool)

    for batch_start in range(0, n, IMAGE_BATCH):
        batch_indices = range(batch_start, min(batch_start + IMAGE_BATCH, n))
        imgs, valid_indices = [], []
        for i in batch_indices:
            try:
                with Image.open(paths[i]) as im:
                    imgs.append(im.convert("RGB"))
                valid_indices.append(i)
            except OSError:
                pass
        if imgs:
            embs = encode_images(imgs)
            for emb, i in zip(embs, valid_indices):
                out[i] = emb
                ok[i] = True
    return out, ok


def fuse_embeddings(title_emb: np.ndarray, image_emb: np.ndarray, image_ok: np.ndarray) -> np.ndarray:
    ok = image_ok[:, None]
    avg = np.where(ok, (title_emb + image_emb) * 0.5, title_emb)
    return avg / np.linalg.norm(avg, axis=1, keepdims=True)

## Цикл извлечения эмбеддингов с чекпоинтингом по шардам

In [7]:
emb_dtype = pl.Float32
n = items.height
n_chunks = (n + CHUNK_SIZE - 1) // CHUNK_SIZE

for ci in tqdm(range(n_chunks), desc="shards"):
    shard_path = SHARD_DIR / f"shard_{ci:05d}.parquet"
    if shard_path.exists():
        continue

    chunk = items.slice(ci * CHUNK_SIZE, CHUNK_SIZE)
    ids = chunk["item_id"].to_list()
    titles = chunk["title"].to_list()
    paths = [image_path(i) for i in ids]

    title_emb = encode_texts_batched(titles)
    image_emb, image_ok = encode_images_batched(paths)
    embedding = fuse_embeddings(title_emb, image_emb, image_ok)

    pl.DataFrame({"item_id": ids}).with_columns(
        pl.Series("embedding", embedding, dtype=pl.Array(emb_dtype, EMB_DIM)),
    ).write_parquet(shard_path)

print("Шардов:", len(list(SHARD_DIR.glob("shard_*.parquet"))))

shards:   0%|          | 0/146 [00:00<?, ?it/s]

Шардов: 146


## Объединение шардов в один файл

In [ ]:
cols = ["item_id", "embedding"]
shards = pl.scan_parquet(SHARD_DIR / "shard_*.parquet").sort("item_id")
shards.select(cols).sink_parquet(FINAL_PATH)
print("Сохранено:", FINAL_PATH)

Строк: 2,903,192
Сохранено: ..\data\embeddings_siglip2\item_embeddings.parquet
